In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np

In [2]:
from pandas import DataFrame
import sceptr
import pandas as pd
import muon as mu

In [3]:
path = r"/ix1/ylee/Yifan_Zhang/Code_data/Tumor/GSE139555_2019/data/Processed/"
filename = "T_DE_per_sample_TCRembvdjdb_singlets.h5mu"
DATA_PATH = path + filename

mdata_ori = mu.read(DATA_PATH)

In [19]:
mdata = mdata_ori.copy()
mdata

MuData object with n_obs × n_vars = 27159 × 2551
  obs:	'isT', 'ident', 'patient', 'source', 'type', 'subtype', 'clone_loc', 'unique_clone_id', 'cloned', 'in_two_tissue', 'clone_status', 'VJ_1_cdr3_aa', 'VJ_1_v_call', 'VJ_1_j_call', 'VDJ_1_cdr3_aa', 'VDJ_1_v_call', 'VDJ_1_j_call', 'VDJ_1_cdr3_aa_length', 'VJ_1_cdr3_aa_length', 'vdjdb_match', 'vdjdb_label'
  uns:	'hvg_union_COMBAT_ID_meta', 'tcr_embs_feature_names'
  obsm:	'VDJ_1_j_call', 'VDJ_1_v_call', 'VJ_1_j_call', 'VJ_1_v_call', 'X_VDJ_1_cdr3_aa_atchley', 'X_VDJ_1_cdr3_aa_composition', 'X_VJ_1_cdr3_aa_atchley', 'X_VJ_1_cdr3_aa_composition', 'tcr_embs'
  2 modalities
    gex:	27159 × 2551
      obs:	'sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'n_counts', 'patient', 'source', 'type', 'subtype', 'clone_status'
      var:	'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'hvg_union_COMBAT_ID', 'mean', 'std'
      uns:	'X_umap_harmony', 'hvg_union_COMBAT_ID_k200', 'hvg_union_COMBAT_ID_meta', 'log1p', 'neighbors', 'neighbors_harmony', 'patient_colors', 'pca', 'source_colors', 'subtype_colors', 'type_colors', 'umap'
      obsm:	'X_pca', 'X_pca_harmony', 'X_umap', 'X_umap_harmony'
      varm:	'PCs'
      obsp:	'connectivities', 'distances', 'neighbors_harmony_connectivities', 'neighbors_harmony_distances'
    airr:	27159 × 0
      obs:	'sample', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size', 'cc_aa_tcrdist', 'cc_aa_tcrdist_size', 'clonal_expansion'
      uns:	'cc_aa_tcrdist', 'chain_indices', 'clone_id', 'ir_dist_aa_identity', 'ir_dist_aa_tcrdist', 'ir_dist_nt_identity'
      obsm:	'airr', 'chain_indices'

In [20]:
clone_df = mdata.obs[["unique_clone_id", "vdjdb_match",
                      "VJ_1_cdr3_aa","VJ_1_v_call","VJ_1_j_call",
            "VDJ_1_cdr3_aa","VDJ_1_v_call","VDJ_1_j_call"]]

# Create a dictionary mapping the old names to the new names
rename_mapping = {
    "VJ_1_cdr3_aa": "cdr3_a_aa",
    "VJ_1_v_call": "v_a_gene",
    "VJ_1_j_call": "j_a_gene",
    "VDJ_1_cdr3_aa": "cdr3_b_aa",
    "VDJ_1_v_call": "v_b_gene",
    "VDJ_1_j_call": "j_b_gene"
}

# Rename the columns
clone_df = clone_df.rename(columns=rename_mapping)
print(clone_df.head())               

                       unique_clone_id  vdjdb_match     cdr3_a_aa  v_a_gene  \
LT1_CACATAGAGTGTACCT-1         Lung1_0        False  ALGSQGGSEKLV   TRAV9-2   
LN3_CTTAGGATCGCGCCAA-2     Lung3_10014        False    ALWLYGNKLV    TRAV16   
LN1_CCTATTAAGTCTCAAC-1      Lung1_1003         True    AMRYTGNQFY  TRAV12-3   
LN3_GTAACTGAGTTAGGTA-2     Lung3_10035         True   AVNNNAGNMLT   TRAV8-1   
LN3_GACGCGTTCAGTGTTG-2     Lung3_10046        False  AVCSGGSNYKLT    TRAV21   

                       j_a_gene       cdr3_b_aa  v_b_gene j_b_gene  
LT1_CACATAGAGTGTACCT-1   TRAJ57  ASSQDQTARSYGYT   TRBV4-1  TRBJ1-2  
LN3_CTTAGGATCGCGCCAA-2   TRAJ47  ASRWGKGARTGELF    TRBV19  TRBJ2-2  
LN1_CCTATTAAGTCTCAAC-1   TRAJ49  ASSSRLAGFTGELF   TRBV5-4  TRBJ2-2  
LN3_GTAACTGAGTTAGGTA-2   TRAJ39   ASSLGGYSNQPQH   TRBV7-9  TRBJ1-5  
LN3_GACGCGTTCAGTGTTG-2   TRAJ53  SAPDRGVPSNYGYT  TRBV20-1  TRBJ1-2  


In [21]:
match_df = clone_df[clone_df["vdjdb_match"] == True]
unmatch_df = clone_df[clone_df["vdjdb_match"] == False]

In [22]:
import logging
import pandas as pd
import tidytcells as tt

logging.basicConfig(level=logging.WARNING)

# ---- 2. Map AIRR/scirpy columns -> TRAV/CDR3A/TRBV/CDR3B (+ J genes) ------
def prefix_c(s):
    """Prepend 'C' to CDR3 aa seqs; leave empty/NaN as ''."""
    s = s.astype(str)
    return ("C" + s + "F").where(s.notna() & (s != "") & (s != "nan"), "")

In [23]:
tcr_all = pd.DataFrame({
    "TRAV":  clone_df["v_a_gene"].values,   # keep raw IMGT symbol (e.g. TRAV1-2*01)
    "TRAJ":  clone_df["j_a_gene"].values,
    "CDR3A": prefix_c(clone_df["cdr3_a_aa"]).values,
    "TRBV":  clone_df["v_b_gene"].values,
    "TRBJ":  clone_df["j_b_gene"].values,
    "CDR3B": prefix_c(clone_df["cdr3_b_aa"]).values,
},  index=clone_df['unique_clone_id'])

# ---- 3. Standardize V/J genes to IMGT gene-level -------------------------
# NOTE 1: set species correctly. "homosapiens" / "musmusculus" / "any".
SPECIES = "homosapiens"
cols = ["TRAV", "TRAJ", "TRBV", "TRBJ"]
tcr_all[cols] = tcr_all[cols].map(
    lambda x: tt.tr.standardize(
        symbol=x,
        species=SPECIES,
        precision="gene",
        enforce_functional=True,   # drop ORF/pseudogenes like TRBV21-1
        on_fail="reject",          # return None on failure
        log_failures=True,
    ),
    na_action="ignore",
)

print(tcr_all.head())

                     TRAV    TRAJ           CDR3A      TRBV     TRBJ  \
unique_clone_id                                                        
Lung1_0           TRAV9-2  TRAJ57  CALGSQGGSEKLVF   TRBV4-1  TRBJ1-2   
Lung3_10014        TRAV16  TRAJ47    CALWLYGNKLVF    TRBV19  TRBJ2-2   
Lung1_1003       TRAV12-3  TRAJ49    CAMRYTGNQFYF   TRBV5-4  TRBJ2-2   
Lung3_10035       TRAV8-1  TRAJ39   CAVNNNAGNMLTF   TRBV7-9  TRBJ1-5   
Lung3_10046        TRAV21  TRAJ53  CAVCSGGSNYKLTF  TRBV20-1  TRBJ1-2   

                            CDR3B  
unique_clone_id                    
Lung1_0          CASSQDQTARSYGYTF  
Lung3_10014      CASRWGKGARTGELFF  
Lung1_1003       CASSSRLAGFTGELFF  
Lung3_10035       CASSLGGYSNQPQHF  
Lung3_10046      CSAPDRGVPSNYGYTF  


In [24]:
tcr_matched = pd.DataFrame({
    "TRAV":  match_df["v_a_gene"].values,   # keep raw IMGT symbol (e.g. TRAV1-2*01)
    "TRAJ":  match_df["j_a_gene"].values,
    "CDR3A": prefix_c(match_df["cdr3_a_aa"]).values,
    "TRBV":  match_df["v_b_gene"].values,
    "TRBJ":  match_df["j_b_gene"].values,
    "CDR3B": prefix_c(match_df["cdr3_b_aa"]).values,
}, index=match_df['unique_clone_id'])

# ---- 3. Standardize V/J genes to IMGT gene-level -------------------------
# NOTE 1: set species correctly. "homosapiens" / "musmusculus" / "any".
SPECIES = "homosapiens"
cols = ["TRAV", "TRAJ", "TRBV", "TRBJ"]
tcr_matched[cols] = tcr_matched[cols].map(
    lambda x: tt.tr.standardize(
        symbol=x,
        species=SPECIES,
        precision="gene",
        enforce_functional=True,   # drop ORF/pseudogenes like TRBV21-1
        on_fail="reject",          # return None on failure
        log_failures=True,
    ),
    na_action="ignore",
)

print(tcr_matched.head())

                         TRAV    TRAJ           CDR3A      TRBV     TRBJ  \
unique_clone_id                                                            
Lung1_1003           TRAV12-3  TRAJ49    CAMRYTGNQFYF   TRBV5-4  TRBJ2-2   
Lung3_10035           TRAV8-1  TRAJ39   CAVNNNAGNMLTF   TRBV7-9  TRBJ1-5   
Lung3_10059             TRAV2  TRAJ43     CAAYNNNDMRF   TRBV5-1  TRBJ2-1   
Lung1_1022       TRAV38-2/DV8  TRAJ44  CAYSLTGTASKLTF  TRBV29-1  TRBJ1-1   
Lung3_10426          TRAV12-2  TRAJ20      CAVSDYKLSF   TRBV5-6  TRBJ1-1   

                             CDR3B  
unique_clone_id                     
Lung1_1003        CASSSRLAGFTGELFF  
Lung3_10035        CASSLGGYSNQPQHF  
Lung3_10059        CASSLAGQVTDEQFF  
Lung1_1022          CSVTTGYMNTEAFF  
Lung3_10426      CASSLGGGPPLNTEAFF  


In [25]:
tcr_unmatched = pd.DataFrame({
    "TRAV":  unmatch_df["v_a_gene"].values,   # keep raw IMGT symbol (e.g. TRAV1-2*01)
    "TRAJ":  unmatch_df["j_a_gene"].values,
    "CDR3A": prefix_c(unmatch_df["cdr3_a_aa"]).values,
    "TRBV":  unmatch_df["v_b_gene"].values,
    "TRBJ":  unmatch_df["j_b_gene"].values,
    "CDR3B": prefix_c(unmatch_df["cdr3_b_aa"]).values,
},index=unmatch_df['unique_clone_id'])

# ---- 3. Standardize V/J genes to IMGT gene-level -------------------------
# NOTE 1: set species correctly. "homosapiens" / "musmusculus" / "any".
SPECIES = "homosapiens"

cols = ["TRAV", "TRAJ", "TRBV", "TRBJ"]
tcr_unmatched[cols] = tcr_unmatched[cols].map(
    lambda x: tt.tr.standardize(
        symbol=x,
        species=SPECIES,
        precision="gene",
        enforce_functional=True,   # drop ORF/pseudogenes like TRBV21-1
        on_fail="reject",          # return None on failure
        log_failures=True,
    ),
    na_action="ignore",
)

print(tcr_unmatched.head())

                         TRAV    TRAJ              CDR3A      TRBV     TRBJ  \
unique_clone_id                                                               
Lung1_0               TRAV9-2  TRAJ57     CALGSQGGSEKLVF   TRBV4-1  TRBJ1-2   
Lung3_10014            TRAV16  TRAJ47       CALWLYGNKLVF    TRBV19  TRBJ2-2   
Lung3_10046            TRAV21  TRAJ53     CAVCSGGSNYKLTF  TRBV20-1  TRBJ1-2   
Lung1_1006       TRAV38-2/DV8  TRAJ45     CAYATGGGADGLTF  TRBV12-4  TRBJ2-7   
Lung3_10065          TRAV26-1  TRAJ45  CIVRVRRAGGGADGLTF   TRBV5-1  TRBJ2-3   

                            CDR3B  
unique_clone_id                    
Lung1_0          CASSQDQTARSYGYTF  
Lung3_10014      CASRWGKGARTGELFF  
Lung3_10046      CSAPDRGVPSNYGYTF  
Lung1_1006       CASSSSGGRFLYEQYF  
Lung3_10065       CASSPGHRDLDTQYF  


## directly operate on sceptr’s TCR representations

In [26]:
reps = sceptr.calc_vector_representations(tcr_all)
reps.shape

(27159, 64)

In [27]:
reps_df = pd.DataFrame(reps, index=tcr_all.index)

In [28]:
reps_df.head(5)

,0,1,2,3,4,5,6,7,8,9,...,54,55,56,57,58,59,60,61,62,63
unique_clone_id,,,,,,,,,,,,,,,,,,,,,
Lung1_0,-0.066062,0.203189,-0.119915,0.255406,0.010583,0.074428,0.026460,-0.064485,0.084508,0.034743,...,-0.196567,-0.074838,-0.105573,-0.016631,0.115718,-0.164589,-0.232779,0.045374,-0.041028,0.040143
Lung3_10014,0.030877,0.084759,-0.008068,-0.013641,-0.091246,0.022414,0.141316,0.006437,0.053522,-0.020711,...,0.045426,-0.017274,0.142034,0.088398,0.082415,-0.315488,0.065662,-0.087076,0.008372,-0.151970
Lung1_1003,0.090413,0.128990,0.063036,-0.039741,-0.071908,0.050091,-0.099933,-0.184146,0.097376,-0.109032,...,-0.253621,-0.158263,0.201164,0.047315,-0.229418,0.042264,0.261577,0.075096,-0.084832,0.200335
Lung3_10035,0.128852,0.074536,-0.085337,0.231217,-0.040411,-0.026651,0.210416,-0.018255,-0.104617,0.145407,...,-0.140525,-0.036755,0.004123,-0.011947,-0.011308,0.122158,-0.031237,0.027552,-0.080475,0.053137
Lung3_10046,0.114194,-0.075526,-0.098779,0.084671,0.234572,-0.060513,-0.006399,-0.210198,0.010956,-0.120321,...,0.174256,-0.238831,-0.155532,-0.122914,-0.177899,-0.154115,-0.077941,0.039863,-0.003492,-0.153735


In [30]:
reps_df.to_csv('TCR_sceptrEmb_perclone.csv')

## cross-distance matrix between two sets of TCR

In [ ]:
aa

In [ ]:
cdist_matrix = sceptr.calc_cdist_matrix(tcr_matched, tcr_unmatched)
print(cdist_matrix)

# Unused functions

In [ ]:
aa

In [ ]:
### model variants
from sceptr import variant
sceptr_tiny = variant.tiny()
tiny_reps = sceptr_tiny.calc_vector_representations(tcrs)
print(tiny_reps.shape)

In [ ]:
### within-set distances
pdist_vector = sceptr.calc_pdist_vector(tcrs)
print(pdist_vector)
## return shape 0.5*N*(N-1)